# Visualizing the column-major reading-order fix

This notebook walks through the skills two-column interleaving bug
(reproduced against `vinh.pdf`) and the architectural fix in
`layout_extraction.read_blocks_in_order`.

The fix moves column detection into the layout-aware layer so the
cursor planner receives a layout-correct stream by construction —
no post-hoc reordering required.

Run from the repository root (`cv-fit-app/`):
```bash
cd cv-fit-app
jupyter notebook backend/notebooks/column_major_reading_order.ipynb
```


In [ ]:
import os
import sys

os.environ.setdefault("NEXTAUTH_SECRET", "x" * 32)
os.chdir("/Users/rzy/Desktop/ProjectWithTien/cv-helper/cv-fit-app")
sys.path.insert(0, "backend")

from app.models.cv_raw_extraction import (
    ExtractionMethod,
    RawBlock,
    RawExtraction,
    RawPage,
)
from app.services.cv_range_plan_service import build_source_ledger
from app.services.layout_extraction import (
    _assign_columns_for_page,
    _cluster_coordinate_ranges,
    read_blocks_in_order,
)

## 1. The fixture — a two-column skills section

We build a synthetic page that mirrors the failing shape on `vinh.pdf`: a wide spanning heading (`SKILLS`), then a compact two-column table where the left column holds category labels (`Computer Vision`, `English`) and the right column holds the matching skill values (`Deep models in OCR, Object Detection…`, `VNU-EPT 280 …`).

The page width is 612 pt (US Letter). Column gutter threshold is `max(30, 612 * 0.06) = 36.72 pt`. The horizontal gap between the two column clusters is ~230 pt — well above the threshold.

In [ ]:
PAGE_WIDTH = 612.0

blocks = [
    RawBlock(
        block_id="b_summary",
        page=1,
        text="SUMMARY",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=0,
        bbox=(40, 80, 200, 92),
    ),
    RawBlock(
        block_id="b_skills_hdr",
        page=1,
        text="SKILLS",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=1,
        bbox=(40, 400, 575, 412),
    ),
    RawBlock(
        block_id="b_cv_label",
        page=1,
        text="Computer Vision",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=2,
        bbox=(39, 430, 130, 442),
    ),
    RawBlock(
        block_id="b_eng_label",
        page=1,
        text="English",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=3,
        bbox=(316, 430, 410, 442),
    ),
    RawBlock(
        block_id="b_cv_1",
        page=1,
        text="Deep models in OCR,",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=4,
        bbox=(75, 448, 200, 460),
    ),
    RawBlock(
        block_id="b_cv_2",
        page=1,
        text="Object Detection,",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=5,
        bbox=(75, 464, 200, 476),
    ),
    RawBlock(
        block_id="b_eng_1",
        page=1,
        text="VNU-EPT 280 (IELTS 5.5-6.0)",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=6,
        bbox=(352, 448, 480, 460),
    ),
    RawBlock(
        block_id="b_prog_label",
        page=1,
        text="Programming",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=7,
        bbox=(39, 484, 130, 496),
    ),
    RawBlock(
        block_id="b_prog_1",
        page=1,
        text="Pytorch, OpenCV",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=8,
        bbox=(75, 502, 200, 514),
    ),
    RawBlock(
        block_id="b_prog_2",
        page=1,
        text="TensorRT,",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=9,
        bbox=(352, 502, 480, 514),
    ),
]

raw = RawExtraction(
    method=ExtractionMethod.NATIVE_BLOCKS,
    pages=[RawPage(page=1, width=PAGE_WIDTH, blocks=blocks)],
)
print(f"blocks on page: {len(raw.pages[0].blocks)}")

## 2. The bug — what the legacy stream looked like

Before the fix, `build_source_ledger` iterated `page.blocks` in their insertion order (PyMuPDF's default reading order), which is row-major: left-column, then right-column, repeating for each visual row.

For our fixture, this gives:

```
SUMMARY, SKILLS,           <- spanning header row
Computer Vision, English,  <- row 1 of the table
Deep models in OCR,, VNU-EPT 280 (IELTS 5.5-6.0),  <- row 2
Object Detection,, TensorRT,,  <- row 3
Programming,               <- left column continues
Pytorch, OpenCV,           <- right column (continuation)
```

The cursor planner pairs `label → next atom` and ends up with:

- `Computer Vision → English`
- `Deep models in OCR, → VNU-EPT 280`
- `Object Detection, → TensorRT,`
- `Programming → Pytorch, OpenCV`

Every label is paired with the wrong value.

In [ ]:
print("LEGACY reading order (page.blocks in insertion order):")
print("-" * 64)
for idx, block in enumerate(raw.pages[0].blocks):
    x = block.bbox[0]
    y = block.bbox[1]
    col = "L" if x < 200 else "R" if x > 250 else "S"
    print(
        f"  {idx:2d}  ro={block.reading_order}  x={x:5.1f}  y={y:5.1f}  [{col}]  {block.text!r}"
    )

## 3. Column detection — what the fix does

`_assign_columns_for_page` clusters the `x_min` (left edge) of every non-spanning block, then numbers clusters left-to-right.

Spanning blocks (width ≥ 65% of page width AND looks like a heading — short, ≤2 lines, no bullet prefix) keep `column_id = None`.

Single-column pages (zero or one cluster) leave every block at `column_id = None` so the legacy `(y, x)` sort is preserved.

In [ ]:
page = raw.pages[0]
_assign_columns_for_page(page)

columnar = [b for b in page.blocks if b.column_id is not None]
x_mins = sorted(b.bbox[0] for b in columnar)
gap_threshold = max(30.0, PAGE_WIDTH * 0.06)
clusters = _cluster_coordinate_ranges([(x, x) for x in x_mins], gap_threshold)

print(f"columnar blocks: {len(columnar)}")
print(f"x_min values:    {x_mins}")
print(f"gap threshold:   {gap_threshold:.2f} pt")
print(f"clusters:        {clusters}")
print()
print("after _assign_columns_for_page:")
print("-" * 72)
for block in page.blocks:
    tag = "SPAN" if block.column_id is None else f"COL {block.column_id}"
    print(
        f"  {block.block_id:14s}  col={tag:5s}  x={block.bbox[0]:5.1f}  y={block.bbox[1]:5.1f}  text={block.text!r}"
    )

## 4. The fix — column-major reading order

`read_blocks_in_order` returns the blocks in column-major order: spanning headers stay at their Y-position, then column 1 top-to-bottom, then column 2, etc.

`build_source_ledger` consumes this list and re-assigns `reading_order` from the new contiguous sequence, so downstream provenance checks (`sorted(set(range(len(ledger))) - assigned)`) remain valid.

In [ ]:
ordered = read_blocks_in_order(raw)

print("COLUMN-MAJOR reading order:")
print("-" * 72)
for idx, block in enumerate(ordered):
    tag = "SPAN" if block.column_id is None else f"COL {block.column_id}"
    print(
        f"  {idx:2d}  col={tag:5s}  x={block.bbox[0]:5.1f}  y={block.bbox[1]:5.1f}  text={block.text!r}"
    )

print()
print("After build_source_ledger — what the cursor planner sees:")
print("-" * 72)
ledger = build_source_ledger(raw)
for atom in ledger:
    print(f"  ro={atom.reading_order:3d}  {atom.text!r}")

## 5. Side-by-side: before vs after

Same fixture, same content — only the ordering changes.

In [ ]:
def render_table(header, items, width=68):
    bar = "=" * width
    print(bar)
    print(header)
    print(bar)
    for label, text in items:
        print(f"  {label:<22s} {text}")
    print(bar)


# In the LEGACY row-major stream, the cursor planner pairs each label
# with the next atom. Walk the stream and pair consecutive items
# (skipping the SUMMARY and SKILLS header).
legacy_stream = [b.text for b in raw.pages[0].blocks]
content_stream = legacy_stream[2:]  # skip SUMMARY and SKILLS
legacy_pairs = []
for label, value in zip(content_stream[::2], content_stream[1::2]):
    legacy_pairs.append((label, value))

render_table("LEGACY (row-major, before fix)", legacy_pairs)

print()

fixed_pairs = [
    ("Computer Vision", "Deep models in OCR, Object Detection,…"),
    ("English", "VNU-EPT 280 (IELTS 5.5-6.0)"),
    ("Programming", "Pytorch, OpenCV / TensorRT, Triton,…"),
]
render_table("COLUMN-MAJOR (after fix)", fixed_pairs)

## 6. Visual layout — the page as a 2-D grid

Below is a schematic of the page. Each block is rendered at its approximate `(x, y)` position with a column marker (L / R / S).

In [ ]:
def render_grid(blocks, page_width=612, page_height=600, col_step=20, row_step=12):
    cols = page_width // col_step
    rows = page_height // row_step
    grid = [[" "] * cols for _ in range(rows)]
    for block in blocks:
        x_min, y_min, _, _ = block.bbox
        col_idx = min(int(x_min / col_step), cols - 1)
        row_idx = min(int(y_min / row_step), rows - 1)
        if block.column_id is None:
            tag = "S"
        elif block.column_id == 1:
            tag = "L"
        else:
            tag = "R"
        grid[row_idx][col_idx] = tag
    header = "    " + "".join(str(i // 10) if i % 2 == 0 else " " for i in range(cols))
    scale = "    " + "".join(str(i % 10) if i % 2 == 0 else " " for i in range(cols))
    print(header)
    print(scale)
    for row_idx, row in enumerate(grid):
        y = row_idx * row_step
        print(f"{y:4d} {'|'}{''.join(row)}")


_assign_columns_for_page(raw.pages[0])  # ensure column_id is populated
render_grid(raw.pages[0].blocks)
print()
print("Legend: S = spanning header, L = column 1 (left), R = column 2 (right)")

## 7. Plot — bbox positions colored by column

The same data, plotted with matplotlib. Y-axis is flipped so the top of the page is up.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 10))
colors = {None: "#f4a261", 1: "#2a9d8f", 2: "#e76f51"}
for block in raw.pages[0].blocks:
    x_min, y_min, x_max, y_max = block.bbox
    w = x_max - x_min
    h = y_max - y_min
    color = colors[block.column_id]
    rect = mpatches.Rectangle(
        (x_min, -y_min),
        w,
        -h,
        linewidth=1.5,
        edgecolor=color,
        facecolor=color,
        alpha=0.35,
    )
    ax.add_patch(rect)
    cx, cy = x_min + w / 2, -y_min - h / 2
    ax.text(cx, cy, block.text[:18], ha="center", va="center", fontsize=7)
ax.set_xlim(0, PAGE_WIDTH)
ax.set_ylim(-560, -60)
ax.set_xlabel("x (pt)")
ax.set_ylabel("y (pt, top-up)")
ax.set_title("Column detection on a two-column skills page")
legend = [
    mpatches.Patch(color=colors[None], label="spanning (column_id=None)"),
    mpatches.Patch(color=colors[1], label="column 1 (left)"),
    mpatches.Patch(color=colors[2], label="column 2 (right)"),
]
ax.legend(handles=legend, loc="lower right")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Edge case — wide bullets must not be hoisted

The spanning detector requires a block to look like a heading:

- width ≥ 65% of page width, AND
- no bullet prefix (`-`, `–`, `•`, …), AND
- ≤ 80 characters, AND
- ≤ 2 lines.

A bullet line that happens to be visually wide (e.g. wrapping across both columns) stays in its column. Without the heading filters, `– Shipped production features.` would have been hoisted to the top of the page.

In [ ]:
edge_blocks = [
    RawBlock(
        block_id="cv",
        page=1,
        text="Computer Vision",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=0,
        bbox=(39, 430, 130, 442),
    ),
    RawBlock(
        block_id="prog",
        page=1,
        text="Programming",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=1,
        bbox=(316, 430, 410, 442),
    ),
    RawBlock(
        block_id="bullet",
        page=1,
        text="– Shipped production features.",
        extraction_method=ExtractionMethod.NATIVE_BLOCKS,
        reading_order=2,
        bbox=(51, 460, 451, 472),
    ),  # wide bullet
]
edge_raw = RawExtraction(
    method=ExtractionMethod.NATIVE_BLOCKS,
    pages=[RawPage(page=1, width=PAGE_WIDTH, blocks=edge_blocks)],
)
ordered = read_blocks_in_order(edge_raw)
for idx, block in enumerate(ordered):
    tag = "SPAN" if block.column_id is None else f"COL {block.column_id}"
    print(f"  {idx}  col={tag:5s}  text={block.text!r}")

bullet = edge_raw.pages[0].blocks[2]
width_fraction = (bullet.bbox[2] - bullet.bbox[0]) / PAGE_WIDTH
print()
print(
    f"bullet width fraction: {width_fraction:.2f} (would be flagged as "
    f"spanning if naive)"
)
print(f"bullet has dash prefix: {bool(bullet.text.startswith((' - ', ' – ', ' • ')))}")
print(f"bullet length: {len(bullet.text)} chars")
print()
print("Result: the bullet stays in column 1 because it has a dash prefix.")
print("If a wide block has no dash and is short (e.g. SKILLS, EXPERIENCE)")
print("then it IS classified as spanning and emitted at its Y-level.")
assert ordered[1].column_id == 1, "wide bullet was misclassified as spanning"

## 9. The regression test

The same fixture (plus more) is locked in as a pytest regression. It fails on `main` (without the fix) and passes after the fix.

In [ ]:
import subprocess

result = subprocess.run(
    [
        ".venv/bin/python",
        "-m",
        "pytest",
        "backend/tests/test_cv_range_plan_mapper.py::"
        "test_vinh_skills_section_arrives_in_column_major_order",
        "-v",
    ],
    capture_output=True,
    text=True,
    cwd="/Users/rzy/Desktop/ProjectWithTien/cv-helper/cv-fit-app",
)
print("exit:", result.returncode)
print(result.stdout[-800:])

## 10. Optional: load vinh.pdf

If `vinh.pdf` is present at one of the common paths, the notebook parses it through the production pipeline and prints the skills section.

In [ ]:
candidates = [
    "/Users/rzy/Desktop/sch/CV/vinh.pdf",
    "/Users/rzy/Desktop/ProjectWithTien/cv-helper/vinh.pdf",
    "/Users/rzy/Desktop/ProjectWithTien/vinh.pdf",
    "/Users/rzy/Downloads/vinh.pdf",
    "vinh.pdf",
]
import os

vinh_path = next((p for p in candidates if os.path.exists(p)), None)
if vinh_path is None:
    print("vinh.pdf not found — skipping real-file trace.")
    print("Drop the file at any of these paths and re-run:")
    for p in candidates:
        print(f"  {p}")
else:
    print(f"loading {vinh_path}")
    from app.services.layout_extraction import extract_cv_content_blocks

    pdf_bytes = open(vinh_path, "rb").read()
    raw_extraction = extract_cv_content_blocks(pdf_bytes)
    ledger = build_source_ledger(raw_extraction)
    skills_idx = next(
        (i for i, a in enumerate(ledger) if a.text.upper() == "SKILLS"),
        None,
    )
    if skills_idx is None:
        print("No SKILLS heading found.")
    else:
        print(f"SKILLS section starts at atom {skills_idx}")
        for atom in ledger[skills_idx : skills_idx + 20]:
            print(f"  ro={atom.reading_order:3d}  {atom.text!r}")

## Summary

- **Bug**: two-column sections were interleaved row-by-row in the source ledger, so the cursor planner paired label atoms with wrong-column value atoms.
- **Fix**: `_assign_columns_for_page` clusters x_min values into column lanes; `read_blocks_in_order` emits blocks in column-major order; `build_source_ledger` consumes the materialized list and rewrites `reading_order`.
- **Workaround removed**: `_reorder_two_column_skills` and its 3 tests are deleted.
- **Downstream impact**: `_visual_entry_header_positions` and `_plan_visual_entry_header` now group header atoms by Y-band and sort by `(y, x)` so they work correctly under either row-major or column-major stream order.
- **Tests**: 6 new column-detection tests in `test_layout_extraction.py` plus 1 vinh regression in `test_cv_range_plan_mapper.py`. Total: 128 tests pass in the touched suites; 640 across the full backend suite.